
# Multi-Region Calibrated Weighted Score-Level Late Fusion

**Amaç:** Kaş, göz ve ağız bölgelerinde aynı temel yaklaşım ile eğitilmiş bölgesel modellerin test çıktılarından tek bir **REAL / FAKE** kararı üretmek.

**Seçilen füzyon yöntemi:**  
**Calibrated Weighted Score-Level Late Fusion**

Akış:

`Eyebrow model score -> calibration -> P(fake)_eyebrow`  
`Eye model score -> calibration -> P(fake)_eye`  
`Mouth model score -> calibration -> P(fake)_mouth`  
`Validation-derived weights -> weighted fusion -> final P(fake) -> REAL / FAKE`

Bu notebook, ekip ML standardına göre:
- test setini ağırlık veya kalibrasyon öğrenmek için kullanmaz,
- kalibrasyon ve bölgesel güvenilirlik ağırlıklarını yalnızca validation verisinden üretir,
- kaynak video kimliği üzerinden strict alignment yapar,
- veri/çıktı muhasebesi ve hash tabanlı audit üretir,
- `config_resolved.yaml`, loglar, metrikler, tahminler, grafikler ve manifest kaydeder,
- grafikleri İngilizce ve kısa kenarı en az 600 px olacak şekilde üretir,
- geçersiz/uydurma eşleştirme yapmaz.

> **Kritik:** Üç bölgenin aynı kaynak videoyu temsil eden kayıtları eşleşmek zorundadır. Notebook pozisyona/index sırasına göre eşleştirme yapmaz.



## Drive path + eye provenance correction

Bu sürüm iki şeyi birlikte çözer:

1. AISC paylaşımlı klasörünü Colab'da `.shortcut-targets-by-id` üzerinden çözer.
2. Göz modelinin test çıktısında kaybolmuş olan gerçek `source_video` kimliğini,
   **Deney 1 Frame/secim_metadata.csv** dosyasındaki `dosya_adi -> orijinal_yol`
   eşlemesinden geri yükler.

Bu bir tahmin/varsayım değildir. `secim_metadata.csv`, örneğin
`fake_test_00000.jpg` dosyasının hangi orijinal video klasöründen geldiğini
açıkça kaydeden provenance tablosudur.

Notebook satır sırasına göre eşleştirme yapmaz.


In [ ]:

# 1) Reproducible environment gate
# Saved regional sklearn artifacts were produced with scikit-learn 1.6.1.
# Install the compatible version BEFORE importing sklearn/joblib models.

import importlib.metadata as md
import subprocess
import sys

REQUIRED_SKLEARN = "1.6.1"

try:
    current_sklearn = md.version("scikit-learn")
except md.PackageNotFoundError:
    current_sklearn = None

if current_sklearn != REQUIRED_SKLEARN:
    print(f"Installing scikit-learn=={REQUIRED_SKLEARN} (current: {current_sklearn})")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", f"scikit-learn=={REQUIRED_SKLEARN}"]
    )

print("Environment gate passed.")


Environment gate passed.


In [ ]:

# 2) Mount Google Drive and imports

from google.colab import drive
drive.mount("/content/drive")

import hashlib
import json
import logging
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import yaml
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)



## 3) Tek doğruluk kaynağı: YAML konfigürasyonu

Aşağıdaki blok bu notebook'un tek ayar kaynağıdır. Hiperparametreler kodun farklı yerlerine gömülmez.

**Kaynak deneyler:**
- Nazlıcan / Kaş
- Kader / Göz
- Dilara / Ağız

**Hedef klasör:**
`AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/füzyon sonuçları/<run_id>/`


In [ ]:
# 3) SSOT configuration

CONFIG_YAML = r"""
seed: 42

drive:
  shared_folder_id: "1qDJf3MYGlyKCNCfp7hS82eXp9H8F1Br1"
  shared_folder_name: "AISC DeepFake Çalışmaları"

provenance:
  selection_metadata: "Deney 1/Deney 1 Frame/secim_metadata.csv"

source_runs:
  eyebrow:
    relative_run_dir: "Deney 1/Nazlıcan/Deney 1/Sonuçlar/VGG16 (sıfırdan eğitim) + HOG + GIST + RBF SVM"
    test_frame_predictions: "predictions/test_predictions_frame_level.csv"
    validation_features: "artifacts/features_val.npz"
    preprocessing: "artifacts/feature_preprocessing.joblib"
    svm_model: "artifacts/svm_model.joblib"
    score_type: "decision_score"

  eye:
    relative_run_dir: "Deney 1/Kader/Deney 1/Sonuçlar/20260806_170247_eye_combined_vgg16scratch_hog_gist_rbf_svm_seed42"
    test_frame_predictions: "predictions/test_predictions_frame_level.csv"
    validation_features: "artifacts/features_val.npz"
    preprocessing: "artifacts/feature_preprocessing.joblib"
    svm_model: "artifacts/svm_model.joblib"
    score_type: "decision_score"

  mouth:
    relative_run_dir: "Deney 1/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42"
    test_frame_predictions: "predictions/frame_level_test_predictions.csv"
    validation_features: "artifacts/pca_features.npz"
    svm_model: "artifacts/svm_pipeline/rbf_svm.joblib"
    score_type: "native_probability"

output:
  relative_root: "Deney 1/Nazlıcan/Deney 1/füzyon sonuçları"
  region_name: "multi_region"
  method_name: "calibrated_weighted_score_fusion"

calibration:
  method: "logistic"
  logistic_c: 1.0
  probability_clip_epsilon: 0.000001

weighting:
  strategy: "validation_auc_reliability"
  reliability_floor: 0.000001

fusion:
  frame_to_video_aggregation: "mean"
  final_threshold: 0.5
  strict_alignment: true
  minimum_common_videos: 20
  require_both_classes: true

figures:
  dpi: 150
  min_short_edge_px: 600
"""

CFG = yaml.safe_load(CONFIG_YAML)
SEED = int(CFG["seed"])

random.seed(SEED)
np.random.seed(SEED)

print(yaml.safe_dump(CFG, allow_unicode=True, sort_keys=False))


In [ ]:

# 4) Utility functions: atomic writes, hashes, canonical IDs, metrics

def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")


def atomic_write_text(text: str, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(target.suffix + ".tmp")
    temp.write_text(text, encoding="utf-8")
    if not temp.exists() or temp.stat().st_size == 0:
        raise RuntimeError(f"Atomic text write validation failed: {temp}")
    os.replace(temp, target)


def atomic_write_json(data: dict, target: Path) -> None:
    atomic_write_text(
        json.dumps(data, ensure_ascii=False, indent=2, default=str),
        target,
    )


def atomic_write_yaml(data: dict, target: Path) -> None:
    atomic_write_text(
        yaml.safe_dump(data, allow_unicode=True, sort_keys=False),
        target,
    )


def atomic_write_csv(df: pd.DataFrame, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(target.suffix + ".tmp")
    df.to_csv(temp, index=False)
    check = pd.read_csv(temp)
    if len(check) != len(df):
        raise RuntimeError(f"Atomic CSV validation failed: {temp}")
    os.replace(temp, target)


def atomic_joblib_dump(obj, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(target.suffix + ".tmp")
    joblib.dump(obj, temp)
    _ = joblib.load(temp)
    os.replace(temp, target)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def canonical_video_id(value: str) -> str:
    """
    Stable canonical source-video name.
    Removes class prefixes and file extension, but NEVER aligns by row position.
    """
    value = str(value).strip()
    value = re.sub(r"^(fake|real)::", "", value, flags=re.IGNORECASE)
    value = Path(value).name
    value = re.sub(r"\.(mp4|avi|mov|mkv|webm)$", "", value, flags=re.IGNORECASE)
    return value.strip().lower()


def choose_target_column(df: pd.DataFrame) -> str:
    for candidate in ("target", "true_label_id", "label_id"):
        if candidate in df.columns:
            return candidate
    if "label" in df.columns:
        return "label"
    if "true_label" in df.columns:
        return "true_label"
    raise ValueError("No supported target/label column found.")


def normalize_binary_target(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()
    mapping = {
        "0": 0,
        "0.0": 0,
        "real": 0,
        "false": 0,
        "1": 1,
        "1.0": 1,
        "fake": 1,
        "true": 1,
    }
    out = s.map(mapping)
    if out.isna().any():
        bad = sorted(s[out.isna()].unique().tolist())[:10]
        raise ValueError(f"Unknown binary target values: {bad}")
    return out.astype(int)


def positive_decision_scores(model, X: np.ndarray) -> np.ndarray:
    score = np.asarray(model.decision_function(X)).reshape(-1)
    classes = list(getattr(model, "classes_", [0, 1]))
    if len(classes) != 2:
        raise ValueError(f"Expected binary model classes, got: {classes}")
    # sklearn binary decision_function is positive toward classes_[1].
    if classes[1] not in (1, "1", "fake"):
        score = -score
    return score


def positive_probability(model, X: np.ndarray) -> np.ndarray:
    if not hasattr(model, "predict_proba"):
        raise ValueError("Model does not expose predict_proba.")
    probs = np.asarray(model.predict_proba(X))
    classes = list(model.classes_)
    positive_candidates = [1, "1", "fake"]
    idx = None
    for candidate in positive_candidates:
        if candidate in classes:
            idx = classes.index(candidate)
            break
    if idx is None:
        raise ValueError(f"Cannot identify fake/positive class from classes_: {classes}")
    return probs[:, idx].astype(float)


def logit(p: np.ndarray, eps: float) -> np.ndarray:
    p = np.clip(np.asarray(p, dtype=float), eps, 1.0 - eps)
    return np.log(p / (1.0 - p))


def compute_binary_metrics(y_true: np.ndarray, p_fake: np.ndarray, threshold: float) -> dict:
    y_true = np.asarray(y_true, dtype=int)
    p_fake = np.asarray(p_fake, dtype=float)
    y_pred = (p_fake >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) else float("nan")

    result = {
        "n_samples": int(len(y_true)),
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "specificity": float(specificity),
        "confusion_matrix": cm.tolist(),
    }

    if len(np.unique(y_true)) == 2:
        result["roc_auc"] = float(roc_auc_score(y_true, p_fake))
        result["average_precision"] = float(average_precision_score(y_true, p_fake))
    else:
        result["roc_auc"] = None
        result["average_precision"] = None

    return result


def verify_figure(path: Path, min_short_edge_px: int) -> None:
    with Image.open(path) as img:
        if min(img.size) < min_short_edge_px:
            raise RuntimeError(
                f"Figure resolution too small: {path.name}, size={img.size}"
            )


def resolve_aisc_root() -> Path:
    """
    Resolve the shared AISC folder without guessing.
    Preferred Colab mount for shared folders is .shortcut-targets-by-id.
    A MyDrive fallback is accepted only if the exact folder already exists.
    """
    shared_id = str(CFG["drive"]["shared_folder_id"])
    folder_name = str(CFG["drive"]["shared_folder_name"])

    candidates = [
        Path("/content/drive/.shortcut-targets-by-id") / shared_id / folder_name,
        Path("/content/drive/MyDrive") / folder_name,
    ]

    existing = [p for p in candidates if p.exists()]
    if not existing:
        diagnostics = []
        for p in candidates:
            diagnostics.append(f"- {p} -> exists={p.exists()}")
        raise FileNotFoundError(
            "AISC shared root could not be resolved.\n"
            + "\n".join(diagnostics)
            + "\nMake sure Google Drive is mounted in this Colab runtime."
        )

    # Prefer the provider-ID path because it is stable for this shared folder.
    for p in existing:
        if ".shortcut-targets-by-id" in str(p):
            return p.resolve()
    return existing[0].resolve()


def resolve_source_paths(aisc_root: Path) -> dict:
    resolved = {}
    for region, spec in CFG["source_runs"].items():
        run_dir = aisc_root / spec["relative_run_dir"]
        paths = {
            "run_dir": run_dir,
            "test_frame_predictions": run_dir / spec["test_frame_predictions"],
            "validation_features": run_dir / spec["validation_features"],
            "svm_model": run_dir / spec["svm_model"],
        }
        if "preprocessing" in spec:
            paths["preprocessing"] = run_dir / spec["preprocessing"]
        resolved[region] = paths

    resolved["_provenance"] = {
        "selection_metadata": aisc_root / CFG["provenance"]["selection_metadata"]
    }
    return resolved



## Preflight: Drive ve kaynak dosya doğrulama

Bu sürümde çıktı klasörü **kaynak dosyalar doğrulanmadan önce oluşturulmaz**. Böylece hatalı bir Drive yolu gereksiz boş klasörler üretmez.

Beklenen ana yol:
`AISC DeepFake Çalışmaları / Deney 1 / <Kişi> / Deney 1 / ...`


In [ ]:
# 5) Resolve shared Drive root and validate ALL source artifacts BEFORE creating outputs

AISC_ROOT = resolve_aisc_root()
ALL_PATHS = resolve_source_paths(AISC_ROOT)

PROVENANCE_PATHS = ALL_PATHS.pop("_provenance")
SOURCE_PATHS = ALL_PATHS

print("Resolved AISC root:")
print(AISC_ROOT)
print()

missing_rows = []

for region, paths in SOURCE_PATHS.items():
    for role, path in paths.items():
        exists = path.exists()
        print(f"[{region:8s}] {role:24s} exists={exists} -> {path}")
        if not exists:
            missing_rows.append(
                {
                    "scope": region,
                    "role": role,
                    "path": str(path),
                }
            )

for role, path in PROVENANCE_PATHS.items():
    exists = path.exists()
    print(f"[{'provenance':8s}] {role:24s} exists={exists} -> {path}")
    if not exists:
        missing_rows.append(
            {
                "scope": "provenance",
                "role": role,
                "path": str(path),
            }
        )

if missing_rows:
    missing_df = pd.DataFrame(missing_rows)
    display(missing_df)
    raise FileNotFoundError(
        "One or more required regional/provenance artifacts are missing. "
        "No fusion output directory has been created. "
        "Check the table above for the exact unresolved path(s)."
    )

print("\nPreflight passed: all required regional and provenance artifacts exist.")


In [ ]:
# 6) Initialize run directory only after successful preflight

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
RUN_ID = (
    f"{timestamp}_"
    f"{CFG['output']['region_name']}_"
    f"{CFG['output']['method_name']}_"
    f"seed{SEED}"
)

OUTPUT_ROOT = AISC_ROOT / CFG["output"]["relative_root"]
if not OUTPUT_ROOT.exists():
    raise FileNotFoundError(
        f"Expected fusion output root does not exist: {OUTPUT_ROOT}\n"
        "The notebook will not silently create a replacement hierarchy."
    )

RUN_DIR = OUTPUT_ROOT / RUN_ID
if RUN_DIR.exists():
    raise FileExistsError(
        f"Run directory already exists and will not be overwritten: {RUN_DIR}"
    )

SUBDIRS = {
    name: RUN_DIR / name
    for name in (
        "checkpoints",
        "logs",
        "metrics",
        "predictions",
        "figures",
        "artifacts",
    )
}

RUN_DIR.mkdir(parents=False, exist_ok=False)
for path in SUBDIRS.values():
    path.mkdir(parents=False, exist_ok=False)

atomic_write_text(
    "No trainable backbone checkpoint is created by weighted score-level late fusion.\n"
    "Regional models remain immutable; calibration objects and weights are stored in artifacts/.\n",
    SUBDIRS["checkpoints"] / "README.txt",
)

LOGGER = logging.getLogger(RUN_ID)
LOGGER.setLevel(logging.INFO)
LOGGER.handlers.clear()

file_handler = logging.FileHandler(
    SUBDIRS["logs"] / "fusion.log",
    mode="w",
    encoding="utf-8",
)
stream_handler = logging.StreamHandler(sys.stdout)

formatter = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
file_handler.setFormatter(formatter)
stream_handler.setFormatter(formatter)
LOGGER.addHandler(file_handler)
LOGGER.addHandler(stream_handler)

resolved_cfg = dict(CFG)
resolved_cfg["run_id"] = RUN_ID
resolved_cfg["created_at"] = now_iso()
resolved_cfg["aisc_root"] = str(AISC_ROOT)
resolved_cfg["run_dir"] = str(RUN_DIR)
resolved_cfg["resolved_source_run_dirs"] = {
    region: str(paths["run_dir"])
    for region, paths in SOURCE_PATHS.items()
}
resolved_cfg["resolved_provenance_paths"] = {
    key: str(path)
    for key, path in PROVENANCE_PATHS.items()
}
atomic_write_yaml(resolved_cfg, RUN_DIR / "config_resolved.yaml")

LOGGER.info("Initialized fusion run: %s", RUN_ID)
LOGGER.info("Output directory: %s", RUN_DIR)


In [ ]:

# 7) Environment / dependency audit

environment = {
    "created_at": now_iso(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "joblib": joblib.__version__,
}

try:
    import torch
    environment["torch"] = torch.__version__
    environment["cuda_available"] = bool(torch.cuda.is_available())
    environment["cuda_device"] = (
        torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    )
except Exception as exc:
    # Environment inspection failure is recorded, not silently ignored.
    environment["torch_inspection_error"] = repr(exc)

atomic_write_json(environment, RUN_DIR / "environment.json")

requirements_text = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"],
    text=True,
)
atomic_write_text(requirements_text, RUN_DIR / "requirements_lock.txt")

LOGGER.info("Environment audit saved.")


In [ ]:
# 8) Input audit manifest: source artifacts and provenance files are read-only

input_manifest_rows = []

for region, paths in SOURCE_PATHS.items():
    for role, path in paths.items():
        if role == "run_dir":
            continue
        input_manifest_rows.append(
            {
                "scope": region,
                "role": role,
                "path": str(path),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

for role, path in PROVENANCE_PATHS.items():
    input_manifest_rows.append(
        {
            "scope": "provenance",
            "role": role,
            "path": str(path),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
    )

input_manifest = pd.DataFrame(input_manifest_rows)
atomic_write_csv(input_manifest, SUBDIRS["artifacts"] / "input_manifest.csv")

LOGGER.info("Source artifacts and provenance metadata validated. No source files will be modified.")
display(input_manifest)



## 8) Validation-only calibration and weight estimation

Bu aşamada **test verisine dokunulmaz**.

- Kaş ve göz için `features_val.npz` + kaydedilmiş preprocessing + RBF SVM kullanılır.
- Ağız için `pca_features.npz` içindeki `X_validation / y_validation` + kaydedilmiş RBF SVM kullanılır.
- Her bölge skoru validation etiketleri ile 1-boyutlu Logistic Calibration'a sokulur.
- Bölge ağırlıkları validation ROC-AUC güvenilirliğinden hesaplanır:

`reliability = max(validation_auc - 0.5, floor)`

ve sonra ağırlıklar toplamı 1 olacak şekilde normalize edilir.

Bu yaklaşım test sonuçlarından ağırlık seçilmesini engeller.


In [ ]:

# 8) Validation score extraction

def load_validation_signal(region: str, spec: dict, paths: dict):
    model = joblib.load(paths["svm_model"])

    if region in ("eyebrow", "eye"):
        data = np.load(paths["validation_features"], allow_pickle=False)

        required_keys = {"hog", "gist", "vgg", "targets"}
        missing = required_keys.difference(data.files)
        if missing:
            raise KeyError(f"{region}: validation NPZ missing keys: {sorted(missing)}")

        pre = joblib.load(paths["preprocessing"])
        expected_order = tuple(pre.get("feature_block_order", ("hog", "gist", "vgg")))
        scalers = pre.get("block_scalers")
        pca = pre.get("pca")

        if scalers is None or pca is None:
            raise ValueError(f"{region}: invalid preprocessing artifact.")

        scaled_blocks = []
        for block in expected_order:
            if block not in data.files:
                raise KeyError(f"{region}: missing feature block '{block}'")
            if block not in scalers:
                raise KeyError(f"{region}: missing scaler for block '{block}'")
            scaled_blocks.append(scalers[block].transform(data[block]))

        X_scaled = np.concatenate(scaled_blocks, axis=1)
        X_val = pca.transform(X_scaled)
        y_val = np.asarray(data["targets"], dtype=int)
        raw_signal = positive_decision_scores(model, X_val)
        signal_kind = "decision_score"

    elif region == "mouth":
        data = np.load(paths["validation_features"], allow_pickle=True)

        required_keys = {"X_validation", "y_validation"}
        missing = required_keys.difference(data.files)
        if missing:
            raise KeyError(f"mouth: validation NPZ missing keys: {sorted(missing)}")

        X_val = np.asarray(data["X_validation"])
        y_val = np.asarray(data["y_validation"], dtype=int)

        native_p = positive_probability(model, X_val)
        raw_signal = logit(
            native_p,
            float(CFG["calibration"]["probability_clip_epsilon"]),
        )
        signal_kind = "logit_native_probability"

    else:
        raise ValueError(f"Unsupported region: {region}")

    if len(raw_signal) != len(y_val):
        raise AssertionError(f"{region}: validation score/label length mismatch.")
    if len(np.unique(y_val)) != 2:
        raise ValueError(f"{region}: validation set must contain both classes.")

    return raw_signal.astype(float), y_val.astype(int), signal_kind


calibration_objects = {}
validation_rows = []

for region, spec in CFG["source_runs"].items():
    raw_signal, y_val, signal_kind = load_validation_signal(
        region,
        spec,
        SOURCE_PATHS[region],
    )

    calibrator = LogisticRegression(
        C=float(CFG["calibration"]["logistic_c"]),
        random_state=SEED,
        solver="lbfgs",
    )
    calibrator.fit(raw_signal.reshape(-1, 1), y_val)

    p_val = calibrator.predict_proba(raw_signal.reshape(-1, 1))[:, 1]
    auc = roc_auc_score(y_val, p_val)
    ap = average_precision_score(y_val, p_val)

    calibration_objects[region] = {
        "calibrator": calibrator,
        "signal_kind": signal_kind,
    }

    validation_rows.append(
        {
            "region": region,
            "n_validation": len(y_val),
            "signal_kind": signal_kind,
            "validation_roc_auc": float(auc),
            "validation_average_precision": float(ap),
        }
    )

validation_performance = pd.DataFrame(validation_rows)

floor = float(CFG["weighting"]["reliability_floor"])
validation_performance["reliability"] = (
    validation_performance["validation_roc_auc"] - 0.5
).clip(lower=floor)

rel_sum = float(validation_performance["reliability"].sum())
if not np.isfinite(rel_sum) or rel_sum <= 0:
    raise RuntimeError("Invalid validation reliability sum.")

validation_performance["weight"] = (
    validation_performance["reliability"] / rel_sum
)

WEIGHTS = dict(
    zip(
        validation_performance["region"],
        validation_performance["weight"],
    )
)

if not math.isclose(sum(WEIGHTS.values()), 1.0, rel_tol=1e-9, abs_tol=1e-9):
    raise AssertionError("Fusion weights do not sum to 1.")

atomic_write_csv(
    validation_performance,
    SUBDIRS["metrics"] / "validation_region_reliability.csv",
)
atomic_write_json(
    {
        "strategy": CFG["weighting"]["strategy"],
        "weights": WEIGHTS,
        "derived_from": "validation_only",
    },
    SUBDIRS["metrics"] / "fusion_weights.json",
)
atomic_joblib_dump(
    calibration_objects,
    SUBDIRS["artifacts"] / "regional_calibrators.joblib",
)

LOGGER.info("Validation-only calibration complete.")
LOGGER.info("Fusion weights: %s", WEIGHTS)
display(validation_performance)



## 9) Test tahminlerini standardize et ve video bazında birleştir

Her bölge için frame-level test skorları ortak `P(fake)` ölçeğine dönüştürülür.  
Ardından aynı `source_video` içindeki frame olasılıkları **mean** ile video düzeyine toplanır.

**Kesinlikle yapılmayanlar:**
- satır sırasına göre eşleştirme,
- frame indeksini başka bir bölgenin videosuymuş gibi varsayma,
- yalnızca `fake_test / real_test` gibi toplulaştırılmış kimliği gerçek video kimliğiymiş gibi kullanma.


In [ ]:
# 9A) Restore eye source-video identity from the original frame-selection provenance

selection_meta_path = PROVENANCE_PATHS["selection_metadata"]
selection_meta = pd.read_csv(selection_meta_path, dtype=str)

required_selection_cols = {"sinif", "split", "orijinal_yol", "yeni_yol", "dosya_adi"}
missing_cols = required_selection_cols.difference(selection_meta.columns)
if missing_cols:
    raise KeyError(
        f"Selection metadata is missing required columns: {sorted(missing_cols)}"
    )

selection_test = selection_meta[
    selection_meta["split"].str.strip().str.lower().eq("test")
].copy()

if selection_test.empty:
    raise RuntimeError("Selection metadata contains no test rows.")

if selection_test["dosya_adi"].duplicated().any():
    duplicates = (
        selection_test.loc[
            selection_test["dosya_adi"].duplicated(keep=False),
            "dosya_adi",
        ]
        .drop_duplicates()
        .head(20)
        .tolist()
    )
    raise AssertionError(
        f"Duplicate test dosya_adi values in selection metadata: {duplicates}"
    )

def extract_original_source_video(original_path: str) -> str:
    original_path = str(original_path).strip()
    parent_name = Path(original_path).parent.name
    if not parent_name:
        raise ValueError(f"Cannot recover source video from: {original_path}")
    return parent_name

def eye_flattened_source_filename(output_path: str) -> str:
    """
    eye output:
      fake_test_00000__face_00.jpg
    -> original flattened input:
      fake_test_00000.jpg
    """
    basename = Path(str(output_path)).name
    stem = re.sub(
        r"__face_\d+\.(jpg|jpeg|png)$",
        "",
        basename,
        flags=re.IGNORECASE,
    )
    if stem == basename:
        stem = Path(basename).stem
    return stem + ".jpg"

selection_test["restored_source_video"] = selection_test["orijinal_yol"].map(
    extract_original_source_video
)
selection_test["expected_label"] = (
    selection_test["sinif"].str.strip().str.lower()
)

EYE_PROVENANCE_MAP = selection_test.set_index("dosya_adi")[
    ["restored_source_video", "expected_label", "orijinal_yol"]
].to_dict(orient="index")

# Audit: show the first deterministic mappings.
provenance_preview = selection_test[
    ["dosya_adi", "expected_label", "orijinal_yol", "restored_source_video"]
].head(20)

atomic_write_csv(
    selection_test[
        ["dosya_adi", "expected_label", "orijinal_yol", "restored_source_video"]
    ],
    SUBDIRS["artifacts"] / "eye_source_video_provenance_map.csv",
)

LOGGER.info(
    "Loaded %d test-frame provenance mappings covering %d unique original videos.",
    len(selection_test),
    selection_test["restored_source_video"].nunique(),
)
display(provenance_preview)


In [ ]:
# 9B) Standardize regional test frame predictions and aggregate to video level

def calibration_input_from_test(region: str, df: pd.DataFrame) -> np.ndarray:
    eps = float(CFG["calibration"]["probability_clip_epsilon"])

    if region in ("eyebrow", "eye"):
        if "decision_score" not in df.columns:
            raise KeyError(f"{region}: decision_score column missing.")
        return pd.to_numeric(df["decision_score"], errors="raise").to_numpy(dtype=float)

    if region == "mouth":
        if "fake_probability" not in df.columns:
            raise KeyError("mouth: fake_probability column missing.")
        native_p = pd.to_numeric(
            df["fake_probability"],
            errors="raise",
        ).to_numpy(dtype=float)
        return logit(native_p, eps)

    raise ValueError(region)


def restore_eye_source_video(df: pd.DataFrame) -> pd.DataFrame:
    """
    Restore the original source-video ID for every eye test sample using
    secim_metadata.csv. No row-order or positional matching is used.
    """
    if "resolved_output_path" not in df.columns:
        raise KeyError(
            "eye: resolved_output_path is required to recover flattened input filename."
        )

    out = df.copy()
    out["flattened_input_filename"] = out["resolved_output_path"].map(
        eye_flattened_source_filename
    )

    mapped = out["flattened_input_filename"].map(EYE_PROVENANCE_MAP)

    if mapped.isna().any():
        missing_files = (
            out.loc[mapped.isna(), "flattened_input_filename"]
            .drop_duplicates()
            .head(20)
            .tolist()
        )
        raise RuntimeError(
            "eye: provenance restoration failed for flattened files: "
            f"{missing_files}"
        )

    out["restored_source_video"] = mapped.map(
        lambda item: item["restored_source_video"]
    )
    out["restored_original_path"] = mapped.map(
        lambda item: item["orijinal_yol"]
    )
    out["restored_expected_label"] = mapped.map(
        lambda item: item["expected_label"]
    )

    if "label" in out.columns:
        current_label = out["label"].astype(str).str.strip().str.lower()
    elif "true_label" in out.columns:
        current_label = out["true_label"].astype(str).str.strip().str.lower()
    else:
        current_label = normalize_binary_target(
            out[choose_target_column(out)]
        ).map({0: "real", 1: "fake"})

    mismatched = current_label.ne(out["restored_expected_label"])
    if mismatched.any():
        examples = out.loc[
            mismatched,
            [
                "flattened_input_filename",
                "restored_expected_label",
            ],
        ].head(20)
        raise AssertionError(
            "eye: class label disagrees with selection provenance.\n"
            + examples.to_string(index=False)
        )

    audit = out[
        [
            "sample_id",
            "flattened_input_filename",
            "restored_source_video",
            "restored_original_path",
            "restored_expected_label",
        ]
    ].copy() if "sample_id" in out.columns else out[
        [
            "flattened_input_filename",
            "restored_source_video",
            "restored_original_path",
            "restored_expected_label",
        ]
    ].copy()

    atomic_write_csv(
        audit,
        SUBDIRS["artifacts"] / "eye_source_video_restoration_audit.csv",
    )

    LOGGER.info(
        "Eye source-video restoration succeeded: %d/%d rows mapped, %d unique videos.",
        len(out),
        len(out),
        out["restored_source_video"].nunique(),
    )
    return out


def build_video_table(region: str) -> pd.DataFrame:
    path = SOURCE_PATHS[region]["test_frame_predictions"]
    df = pd.read_csv(path, dtype=str)

    if region == "eye":
        df = restore_eye_source_video(df)
        source_video_series = df["restored_source_video"].astype(str)
    else:
        if "source_video" not in df.columns:
            raise KeyError(f"{region}: source_video column missing in {path}")
        source_video_series = df["source_video"].astype(str)

    target_col = choose_target_column(df)
    target = normalize_binary_target(df[target_col])

    signal = calibration_input_from_test(region, df)
    calibrator = calibration_objects[region]["calibrator"]
    p_fake = calibrator.predict_proba(signal.reshape(-1, 1))[:, 1]

    standardized = pd.DataFrame(
        {
            "source_video_original": source_video_series,
            "canonical_video": source_video_series.map(canonical_video_id),
            "target": target,
            "p_fake_frame": p_fake,
        }
    )

    standardized["fusion_key"] = (
        standardized["target"].astype(str)
        + "::"
        + standardized["canonical_video"]
    )

    label_counts = standardized.groupby("fusion_key")["target"].nunique()
    if (label_counts > 1).any():
        bad = label_counts[label_counts > 1].index.tolist()[:10]
        raise AssertionError(f"{region}: inconsistent labels inside video keys: {bad}")

    agg = (
        standardized.groupby("fusion_key", as_index=False)
        .agg(
            target=("target", "first"),
            canonical_video=("canonical_video", "first"),
            p_fake=("p_fake_frame", "mean"),
            frame_count=("p_fake_frame", "size"),
            source_video_example=("source_video_original", "first"),
        )
    )

    agg = agg.rename(
        columns={
            "p_fake": f"p_fake_{region}",
            "frame_count": f"frame_count_{region}",
            "source_video_example": f"source_video_{region}",
        }
    )

    atomic_write_csv(
        standardized,
        SUBDIRS["artifacts"] / f"{region}_standardized_frame_scores.csv",
    )
    atomic_write_csv(
        agg,
        SUBDIRS["artifacts"] / f"{region}_video_scores.csv",
    )

    return agg


REGION_VIDEO = {
    region: build_video_table(region)
    for region in ("eyebrow", "eye", "mouth")
}

for region, table in REGION_VIDEO.items():
    LOGGER.info(
        "%s: %d frame-derived video keys",
        region,
        len(table),
    )
    display(table.head())


In [ ]:
# 10) Strict cross-region alignment audit

key_sets = {
    region: set(table["fusion_key"].tolist())
    for region, table in REGION_VIDEO.items()
}

common_keys = set.intersection(*key_sets.values())

audit_summary = {
    "run_id": RUN_ID,
    "created_at": now_iso(),
    "strict_alignment": bool(CFG["fusion"]["strict_alignment"]),
    "eye_source_video_restored_from_provenance": True,
    "provenance_file": str(PROVENANCE_PATHS["selection_metadata"]),
    "video_key_counts": {region: len(keys) for region, keys in key_sets.items()},
    "pairwise_intersections": {
        "eyebrow_eye": len(key_sets["eyebrow"] & key_sets["eye"]),
        "eyebrow_mouth": len(key_sets["eyebrow"] & key_sets["mouth"]),
        "eye_mouth": len(key_sets["eye"] & key_sets["mouth"]),
    },
    "three_way_intersection": len(common_keys),
}

atomic_write_json(
    audit_summary,
    SUBDIRS["artifacts"] / "alignment_audit.json",
)

all_keys = sorted(set.union(*key_sets.values()))
audit_rows = []
for key in all_keys:
    audit_rows.append(
        {
            "fusion_key": key,
            "in_eyebrow": key in key_sets["eyebrow"],
            "in_eye": key in key_sets["eye"],
            "in_mouth": key in key_sets["mouth"],
            "in_all_three": key in common_keys,
        }
    )

alignment_df = pd.DataFrame(audit_rows)
atomic_write_csv(
    alignment_df,
    SUBDIRS["artifacts"] / "alignment_audit.csv",
)

LOGGER.info("Alignment audit: %s", audit_summary)
display(pd.DataFrame([audit_summary["video_key_counts"]]))
display(pd.DataFrame([audit_summary["pairwise_intersections"]]))

minimum_common = int(CFG["fusion"]["minimum_common_videos"])

if len(common_keys) < minimum_common:
    raise RuntimeError(
        f"Only {len(common_keys)} common videos found across all three regions; "
        f"minimum required is {minimum_common}. See artifacts/alignment_audit.csv"
    )

# Every eyebrow video is expected to be represented in the eye/mouth fusion pool
# for this experiment. Fail instead of silently dropping eyebrow test videos.
if len(key_sets["eyebrow"] - common_keys) != 0:
    missing = sorted(key_sets["eyebrow"] - common_keys)[:20]
    raise RuntimeError(
        "Some eyebrow test videos are not available in all three regions. "
        f"Examples: {missing}"
    )

LOGGER.info(
    "Strict alignment gate passed with %d common videos. "
    "Eye identities were restored from selection provenance.",
    len(common_keys),
)


In [ ]:

# 11) Merge aligned regional outputs and perform weighted score-level late fusion

eyebrow = REGION_VIDEO["eyebrow"][
    [
        "fusion_key",
        "target",
        "canonical_video",
        "p_fake_eyebrow",
        "frame_count_eyebrow",
        "source_video_eyebrow",
    ]
].copy()

eye = REGION_VIDEO["eye"][
    [
        "fusion_key",
        "target",
        "p_fake_eye",
        "frame_count_eye",
        "source_video_eye",
    ]
].copy()

mouth = REGION_VIDEO["mouth"][
    [
        "fusion_key",
        "target",
        "p_fake_mouth",
        "frame_count_mouth",
        "source_video_mouth",
    ]
].copy()

merged = eyebrow.merge(
    eye,
    on="fusion_key",
    how="inner",
    suffixes=("_eyebrow_check", "_eye_check"),
).merge(
    mouth,
    on="fusion_key",
    how="inner",
)

# Resolve/verify duplicated target columns.
target_cols = [c for c in merged.columns if c.startswith("target")]
target_matrix = merged[target_cols].astype(int)
if not (target_matrix.nunique(axis=1) == 1).all():
    raise AssertionError("Label disagreement detected across regional models.")

merged["target"] = target_matrix.iloc[:, 0].astype(int)
merged = merged.drop(columns=[c for c in target_cols if c != "target"], errors="ignore")

if CFG["fusion"]["require_both_classes"] and merged["target"].nunique() != 2:
    raise RuntimeError("Aligned fusion test set does not contain both REAL and FAKE classes.")

merged["fusion_score"] = (
    WEIGHTS["eyebrow"] * merged["p_fake_eyebrow"]
    + WEIGHTS["eye"] * merged["p_fake_eye"]
    + WEIGHTS["mouth"] * merged["p_fake_mouth"]
)

threshold = float(CFG["fusion"]["final_threshold"])
merged["prediction"] = (merged["fusion_score"] >= threshold).astype(int)
merged["predicted_label"] = merged["prediction"].map({0: "real", 1: "fake"})
merged["true_label"] = merged["target"].map({0: "real", 1: "fake"})
merged["correct"] = merged["prediction"] == merged["target"]
merged["run_id"] = RUN_ID

prediction_columns = [
    "run_id",
    "fusion_key",
    "canonical_video",
    "target",
    "true_label",
    "p_fake_eyebrow",
    "p_fake_eye",
    "p_fake_mouth",
    "fusion_score",
    "prediction",
    "predicted_label",
    "correct",
    "frame_count_eyebrow",
    "frame_count_eye",
    "frame_count_mouth",
    "source_video_eyebrow",
    "source_video_eye",
    "source_video_mouth",
]
merged = merged[prediction_columns].sort_values("fusion_key").reset_index(drop=True)

atomic_write_csv(
    merged,
    SUBDIRS["predictions"] / "fused_video_predictions.csv",
)

metrics = compute_binary_metrics(
    merged["target"].to_numpy(),
    merged["fusion_score"].to_numpy(),
    threshold=threshold,
)
metrics["fusion_method"] = "calibrated_weighted_score_level_late_fusion"
metrics["weights"] = WEIGHTS
metrics["alignment_common_videos"] = int(len(merged))
metrics["test_used_for_weight_selection"] = False
metrics["test_used_for_calibration"] = False
metrics["test_used_for_threshold_selection"] = False

atomic_write_json(
    metrics,
    SUBDIRS["metrics"] / "fusion_test_metrics.json",
)

LOGGER.info("Fusion completed.")
LOGGER.info("Fusion metrics: %s", metrics)
display(pd.DataFrame([{
    k: v for k, v in metrics.items()
    if k not in ("confusion_matrix", "weights")
}]))
display(merged.head())


In [ ]:

# 12) Publication-quality figures (English, >=600 px short edge)

FIG_DIR = SUBDIRS["figures"]
dpi = int(CFG["figures"]["dpi"])
min_short = int(CFG["figures"]["min_short_edge_px"])

y_true = merged["target"].to_numpy(dtype=int)
p_fusion = merged["fusion_score"].to_numpy(dtype=float)
y_pred = merged["prediction"].to_numpy(dtype=int)

# 12.1 Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
fig, ax = plt.subplots(figsize=(6, 6), dpi=dpi)
im = ax.imshow(cm)
ax.set_title("Multi-Region Fusion Confusion Matrix", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Predicted Class", fontsize=11)
ax.set_ylabel("True Class", fontsize=11)
ax.set_xticks([0, 1], labels=["Real", "Fake"])
ax.set_yticks([0, 1], labels=["Real", "Fake"])

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=13)

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
cm_path = FIG_DIR / f"confusion_matrix_{RUN_ID}.png"
fig.savefig(cm_path, dpi=dpi, bbox_inches="tight")
plt.close(fig)
verify_figure(cm_path, min_short)

# 12.2 ROC Curve
fpr, tpr, _ = roc_curve(y_true, p_fusion)
auc = roc_auc_score(y_true, p_fusion)

fig, ax = plt.subplots(figsize=(10, 6), dpi=dpi)
ax.plot(fpr, tpr, linewidth=2, label=f"Fusion ROC (AUC={auc:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, label="Random")
ax.set_title("Multi-Region Fusion ROC Curve", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.legend(frameon=True, loc="lower right")
ax.grid(True, alpha=0.25)
fig.tight_layout()
roc_path = FIG_DIR / f"roc_curve_{RUN_ID}.png"
fig.savefig(roc_path, dpi=dpi, bbox_inches="tight")
plt.close(fig)
verify_figure(roc_path, min_short)

# 12.3 Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_true, p_fusion)
ap = average_precision_score(y_true, p_fusion)

fig, ax = plt.subplots(figsize=(10, 6), dpi=dpi)
ax.plot(recall, precision, linewidth=2, label=f"Fusion PR (AP={ap:.3f})")
ax.set_title("Multi-Region Fusion Precision-Recall Curve", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Recall", fontsize=11)
ax.set_ylabel("Precision", fontsize=11)
ax.legend(frameon=True, loc="lower left")
ax.grid(True, alpha=0.25)
fig.tight_layout()
pr_path = FIG_DIR / f"precision_recall_curve_{RUN_ID}.png"
fig.savefig(pr_path, dpi=dpi, bbox_inches="tight")
plt.close(fig)
verify_figure(pr_path, min_short)

# 12.4 Validation-derived regional weights
weight_df = validation_performance.sort_values("region").copy()
fig, ax = plt.subplots(figsize=(10, 6), dpi=dpi)
ax.bar(weight_df["region"], weight_df["weight"])
ax.set_title("Validation-Derived Regional Fusion Weights", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Facial Region", fontsize=11)
ax.set_ylabel("Fusion Weight", fontsize=11)
ax.grid(True, axis="y", alpha=0.25)

for x, value in enumerate(weight_df["weight"].to_numpy()):
    ax.text(x, value, f"{value:.3f}", ha="center", va="bottom", fontsize=11)

fig.tight_layout()
weights_path = FIG_DIR / f"regional_weights_{RUN_ID}.png"
fig.savefig(weights_path, dpi=dpi, bbox_inches="tight")
plt.close(fig)
verify_figure(weights_path, min_short)

LOGGER.info("Figures generated and resolution-verified.")


In [ ]:

# 13) Accounting equality, run summary, output manifest

expected_fused = len(common_keys)
actual_fused = len(merged)

if expected_fused != actual_fused:
    raise AssertionError(
        f"Fusion accounting mismatch: expected={expected_fused}, actual={actual_fused}"
    )

if merged["fusion_key"].duplicated().any():
    raise AssertionError("Duplicate fusion_key detected in final predictions.")

if merged[
    ["p_fake_eyebrow", "p_fake_eye", "p_fake_mouth", "fusion_score"]
].isna().any().any():
    raise AssertionError("NaN detected in final probability outputs.")

score_cols = ["p_fake_eyebrow", "p_fake_eye", "p_fake_mouth", "fusion_score"]
for column in score_cols:
    if not merged[column].between(0.0, 1.0).all():
        raise AssertionError(f"Probability range violation in column: {column}")

run_summary = {
    "run_id": RUN_ID,
    "status": "COMPLETED",
    "fusion_method": "Calibrated Weighted Score-Level Late Fusion",
    "source_regions": ["eyebrow", "eye", "mouth"],
    "source_run_dirs": {
        region: str(SOURCE_PATHS[region]["run_dir"])
        for region in ("eyebrow", "eye", "mouth")
    },
    "calibration": {
        "method": CFG["calibration"]["method"],
        "derived_from": "validation_only",
    },
    "weighting": {
        "strategy": CFG["weighting"]["strategy"],
        "weights": WEIGHTS,
        "derived_from": "validation_only",
    },
    "fusion": {
        "aggregation": CFG["fusion"]["frame_to_video_aggregation"],
        "threshold": float(CFG["fusion"]["final_threshold"]),
        "aligned_video_count": int(actual_fused),
    },
    "test_metrics": metrics,
    "provenance_restoration": {
        "eye_source_video_restored": True,
        "method": "dosya_adi -> orijinal_yol parent folder",
        "source_file": str(PROVENANCE_PATHS["selection_metadata"]),
        "row_order_matching_used": False,
    },
    "data_leakage_guards": {
        "test_used_for_calibration": False,
        "test_used_for_weight_selection": False,
        "test_used_for_threshold_selection": False,
        "alignment_by_row_position": False,
        "alignment_key": "target + canonical source_video",
    },
    "completed_at": now_iso(),
}

atomic_write_json(run_summary, RUN_DIR / "run_summary.json")

readme = f"""
# Fusion Run

Run ID: {RUN_ID}

Method:
Calibrated Weighted Score-Level Late Fusion

Regions:
- Eyebrow
- Eye
- Mouth

Final weights:
{json.dumps(WEIGHTS, indent=2)}

Final threshold:
{CFG["fusion"]["final_threshold"]}

Aligned videos:
{actual_fused}

Important safeguards:
- Regional source files were read-only.
- Calibration used validation data only.
- Fusion weights used validation data only.
- Test data was used only for final evaluation.
- Cross-region alignment used source-video identity, never row position.
"""
atomic_write_text(readme.strip() + "\n", RUN_DIR / "README.md")

# Output manifest excludes itself to avoid recursive hashing.
manifest_rows = []
for path in sorted(RUN_DIR.rglob("*")):
    if not path.is_file():
        continue
    if path.name == "output_manifest.csv":
        continue
    manifest_rows.append(
        {
            "relative_path": str(path.relative_to(RUN_DIR)),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
    )

output_manifest = pd.DataFrame(manifest_rows)
atomic_write_csv(output_manifest, RUN_DIR / "output_manifest.csv")

LOGGER.info("Run finalized successfully.")
LOGGER.info("Saved to: %s", RUN_DIR)

print("\nFINAL OUTPUT DIRECTORY:")
print(RUN_DIR)
print("\nFINAL METRICS:")
print(json.dumps(metrics, indent=2, ensure_ascii=False))



## Göz kimliği nasıl düzeltildi?

Göz test çıktısındaki `source_video` alanı yalnızca `fake_test / real_test` olarak
tutulmuştu. Ancak deneyin frame seçim tablosu olan `secim_metadata.csv`,
her `fake_test_XXXXX.jpg / real_test_XXXXX.jpg` dosyasının
`orijinal_yol` değerini saklıyor.

Bu sürüm göz örneğinin çıktı dosya adından flattened frame adını çıkarır ve
`secim_metadata.csv` üzerinden **orijinal video klasörünü** geri yükler.

Örnek mantık:

`fake_test_00000__face_00.jpg`
-> `fake_test_00000.jpg`
-> `secim_metadata.csv`
-> `.../fake_frames/test/Deepfakes_434_438/frame_....jpg`
-> `source_video = Deepfakes_434_438`

Böylece üç bölgenin video-level skorları gerçek kaynak video kimliği üzerinden
eşleştirilir; satır sırası kullanılmaz.
